# 结构化输出——JSON，模式验证，约束解码

你的LLM返回了一个字符串。你的应用需要JSON，这一个差距比模型幻觉击败了更多的生产系统。结构化输出是自然语言与类型数据之间的桥梁。

## 问题描述

你需要的是带指定字段、指定类型和指定值约束的JSON对象，不需要自然语言组成的一段话。

最自然的做法是，提示词中告诉模型“使用JSON回复”。这在大部分时候是有效的，但是也会出现导致你的JSON解析器崩溃的时候。

这并不是一个提示词工程的问题，而是解码的问题。

## 基本概念

### 结构化输出家族

#### 基于提示词

“使用有效的JSON格式回答”。没有强制性，模型一般会遵守，但也有例外。

失败模式：使用Markdown包裹了JSON，前导说明，输出截断，错误的结构。

#### JSON模式

API保证输出是一个有效的JSON。比如OpenAI `response_format: {type: "json_object"}`。输出没有解析错误。

失败模式：模式不匹配，额外的键值对，错误的类型，字段缺失。

#### 约束解码

在生成每个词元的时候，解码器掩去所有可能产生非法输出的词元。约束最强，只可能产生合法输出。

### JSON模式 -- 契约语言

JSON模式是你告诉模型输出必须长什么样子，每个主要的结构化输出系统都使用它。
```JSON
{
  "type": "object",
  "properties": {
    "product": { "type": "string" },
    "price": { "type": "number", "minimum": 0 },
    "in_stock": { "type": "boolean" },
    "categories": {
      "type": "array",
      "items": { "type": "string" }
    }
  },
  "required": ["product", "price", "in_stock"]
}
```

难搞的场景也能处理：对象嵌套、对象数组、枚举、模式匹配（正则等）、组合器。

### Pydantic 模式

在Python中，你只需要定一个Pydantic模型，就可以自动生成JSON模式。
```Python
from pydantic import BaseModel

class Product(BaseModel):
  product : str
  price : float
  in_stock: bool
  categories : list[str] = []
```

OpenAI 的 Instructor 直接接受Pydantic作为输入，如果模型的输出不匹配，会自动进行重试。

### 函数调用/工具使用（Function Calling / Tool Use）

相同问题的不同形式。不直接让模型产生JSON格式，而是定义输入类型参数的工具或者函数。模型输出工具调用请求以及结构化的参数。OpenAI管它叫“Function Calling”，Anthropic 管它叫“Tool use”。

### 常见的失败模式

- 幻觉值。 格式对了，值不对。
- 枚举困惑。  语法正确，输出不在枚举范围内。约束解码可以解决。
- 嵌套深度。 多层嵌套会产生更多的错误。
- 数组长度。 模型产生的数组中对象可能过少，也可能过多。
- 缺省字段遗漏。  对于缺省字段没有用`null`填充。

# 动手编码

In [ ]:
"""四种结构化输出家族对比（DeepSeek via LangChain）

对应前文：基于提示词 / JSON 模式 / 函数调用 / 约束解码。
说明：远端 API 无法做词元级 logits 掩码；此处「约束解码」用 DeepSeek strict
structured output（schema 强制 + Pydantic 校验）作为 API 侧最强约束。
"""
import json
import sys
import time
from pathlib import Path
from typing import Any

sys.path.insert(0, str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter, load_project_env

from langchain_core.tools import tool
from langchain_deepseek import ChatDeepSeek
from pydantic import BaseModel, Field, ValidationError
from rich import print as rprint

load_project_env()


class Product(BaseModel):
    """与前文 JSON Schema / Pydantic 示例对齐。"""

    product: str
    price: float = Field(ge=0)
    in_stock: bool
    categories: list[str]


@tool
def extract_product(
    product: str,
    price: float,
    in_stock: bool,
    categories: list[str],
) -> str:
    """从自然语言描述中提取商品结构化字段。"""
    return "ok"


TASK = (
    "从下面描述提取商品信息。"
    "字段必须是 product / price / in_stock / categories。\n"
    "描述：新品无线耳机 AirPods Pro 2，标价 1899 元，目前缺货，"
    "分类是音频、苹果配件。"
)

llm = ChatDeepSeek(model="deepseek-chat", temperature=0)


def validate_product(payload: Any) -> tuple[bool, str, Product | None]:
    try:
        if isinstance(payload, Product):
            obj = payload
        elif isinstance(payload, dict):
            obj = Product.model_validate(payload)
        elif isinstance(payload, str):
            text = payload.strip()
            if text.startswith("```"):
                text = text.strip("`")
                if text.startswith("json"):
                    text = text[4:]
                text = text.strip()
            obj = Product.model_validate(json.loads(text))
        else:
            return False, f"不支持的类型: {type(payload).__name__}", None
        return True, "Pydantic 校验通过", obj
    except (json.JSONDecodeError, ValidationError, TypeError) as e:
        return False, str(e), None


def run_family(name: str, fn) -> dict:
    t0 = time.perf_counter()
    err = None
    raw = None
    try:
        raw = fn()
    except Exception as e:  # noqa: BLE001 — 对比演示要捕获厂商/解析失败
        err = f"{type(e).__name__}: {e}"
    elapsed_ms = (time.perf_counter() - t0) * 1000
    ok, detail, obj = (False, err, None) if err else validate_product(raw)
    return {
        "family": name,
        "ok": ok,
        "ms": round(elapsed_ms, 1),
        "detail": detail,
        "raw": raw if not isinstance(raw, Product) else raw.model_dump(),
        "parsed": None if obj is None else obj.model_dump(),
    }


# 1) 基于提示词：只靠自然语言约束，无 API 级保证
def family_prompt():
    msg = (
        "请仅用有效 JSON 对象回答，不要 Markdown，不要解释。\n" + TASK
    )
    return llm.invoke(msg).content


# 2) JSON 模式：保证是合法 JSON，不保证字段/类型符合契约
def family_json_mode():
    bound = llm.bind(response_format={"type": "json_object"})
    msg = "返回一个 JSON 对象。\n" + TASK
    return bound.invoke(msg).content


# 3) 函数调用 / Tool Use：模型产出工具参数，参数本身即结构化
def family_tool_use():
    bound = llm.bind_tools([extract_product], tool_choice="extract_product")
    msg = bound.invoke(TASK)
    if not msg.tool_calls:
        raise RuntimeError("模型未产生 tool_calls")
    return msg.tool_calls[0]["args"]


# 4) 约束解码（API 近似）：strict schema + Pydantic
def family_constrained():
    structured = llm.with_structured_output(
        Product,
        method="function_calling",
        strict=True,
    )
    return structured.invoke(TASK)


rows = [
    run_family("1. 基于提示词", family_prompt),
    run_family("2. JSON 模式", family_json_mode),
    run_family("3. 函数调用/Tool Use", family_tool_use),
    run_family("4. 约束解码(strict schema)", family_constrained),
]

with SectionPrinter("结构化输出四家族对比 · DeepSeek"):
    for row in rows:
        status = "PASS" if row["ok"] else "FAIL"
        rprint(f"[bold]{row['family']}[/bold]  [{status}]  {row['ms']} ms")
        rprint(f"  detail: {row['detail']}")
        rprint(f"  raw   : {row['raw']}")
        rprint(f"  parsed: {row['parsed']}")
        print()

print("约束强度（弱 → 强）: 提示词 < JSON模式 < 函数调用 ≈ 约束解码(strict)")
print("失败边界: 提示词可夹杂散文/Markdown；JSON模式保证可解析但不保证 schema；")
print("         函数调用/strict 把契约绑在协议层，再经 Pydantic 做类型校验。")
